# MLP-Residual Stream Interaction

How MLP outputs interact with the residual stream: alignment,
contribution ratio, parallel/perpendicular decomposition, and balance.

## Why This Matters

MLP residual interaction provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_residual_interaction import (
    mlp_residual_alignment, mlp_contribution_ratio,
    mlp_residual_decomposition, mlp_vs_attention_balance,
    mlp_residual_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## MLP-Residual Alignment

Does the MLP reinforce or redirect the residual stream?

In [ ]:
for layer in range(4):
    result = mlp_residual_alignment(model, tokens, layer=layer)
    flag = ' [REINFORCING]' if result['is_reinforcing'] else ''
    print(f"  Layer {layer}: align={result['mean_alignment']:.4f}{flag}")

## MLP Contribution Ratio

How much does MLP contribute relative to the residual?

In [ ]:
result = mlp_contribution_ratio(model, tokens)
print(f"Mean ratio: {result['mean_ratio']:.4f}, max at layer {result['max_ratio_layer']}\n")
for p in result['per_layer']:
    bar = '█' * int(p['contribution_ratio'] * 20)
    print(f"  Layer {p['layer']}: resid={p['residual_norm']:.4f}, "
          f"mlp={p['mlp_norm']:.4f}, ratio={p['contribution_ratio']:.4f} {bar}")

## Parallel/Perpendicular Decomposition

What fraction of MLP output reinforces vs adds new information?

In [ ]:
for layer in range(4):
    result = mlp_residual_decomposition(model, tokens, layer=layer, position=-1)
    print(f"  Layer {layer}: ∥={result['parallel_fraction']:.4f}, "
          f"⊥={result['perpendicular_fraction']:.4f}, "
          f"proj={result['projection_scalar']:.4f}")

## Attention vs MLP Balance

Which component dominates at each layer?

In [ ]:
result = mlp_vs_attention_balance(model, tokens)
print(f"Attn dominant: {result['attn_dominant_layers']} layers")
print(f"MLP dominant: {result['mlp_dominant_layers']} layers\n")
for p in result['per_layer']:
    attn_bar = '█' * int(p['attn_fraction'] * 20)
    mlp_bar = '█' * int(p['mlp_fraction'] * 20)
    print(f"  Layer {p['layer']}: attn={p['attn_fraction']:.2f} {attn_bar}")
    print(f"           mlp ={p['mlp_fraction']:.2f} {mlp_bar}  [{p['dominant']}]")

## MLP-Residual Summary

Combined overview of MLP-residual interaction.

In [ ]:
result = mlp_residual_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: align={p['mlp_alignment']:.4f}, "
          f"mlp={p['mlp_norm']:.4f}, attn={p['attn_norm']:.4f}, "
          f"resid={p['residual_norm']:.4f}, ratio={p['mlp_ratio']:.4f}")